# YOLO11s — Minimum Validation Loss benchmark (VGU)

**Current reported YOLO11s:** checkpoint epoch **37/50** (`epoch36.pt`) chosen by **minimum validation loss (box+cls+dfl)** on **255 validation images** from 50 frozen checkpoints.

**Retrospective test results:** COCO mAP@[.50:.95] = **0.7031666802**, AP50 = **0.9240568340**, AP75 = **0.8399927545** on 639 test images (2,461 detections).

**Run modes:** `review` (default; no Drive/GPU needed), `select` (re-verify/resume on original 50 checkpoints, no retraining), `reproduce` (train if missing; requires GPU and prepared data). Do not run `select` on someone else's Drive path.

**Integrity caveat:** historical split may have scene-level overlap; this test was previously viewed under an older checkpoint selection rule. It is not a new prospective hold-out. The prior **max-validation-AP pilot** selecting epoch 36 is archived separately under `results/YOLO11s/max_val_ap_pilot/`.


## 1 · Runtime and data paths

Use `MODE = "select"` to run the correction on the existing 50-epoch Drive run. `review` does not recalculate loss. `reproduce` is for rerunning training on a new or interrupted run.

In [ ]:
from pathlib import Path

# REVIEW (default) displays the archived experiment without a GPU or Drive.
# REPRODUCE trains/evaluates the historical 50-epoch pilot on a Colab GPU.
MODE = "review"   # Default for GitHub readers; use "select" for archived GPU checkpoint verification

RUN_NAME = "yolo11s_colab_pilot_01"  # preserve this name for the archived run
BATCH = 8
DATASET = Path("/content/drive/MyDrive/model_ready")
RUN = (Path("/content/drive/MyDrive/VGU_TrafficSign_Project/yolo11s_colab_runs")
       / RUN_NAME)
WEIGHTS = Path("/content/vgu-yolo11s-weights/yolo11s.pt")

if MODE not in ("review", "select", "reproduce"):
    raise ValueError("MODE must be 'review', 'select' or 'reproduce'")
if BATCH not in (4, 8):
    raise ValueError("BATCH must be 4 or 8")
if MODE in ("select", "reproduce"):
    from google.colab import drive
    drive.mount("/content/drive")
    if not (DATASET / "annotations" / "instances_train.json").is_file():
        raise FileNotFoundError(
            "Prepared dataset not found. Update DATASET to your own Drive copy "
            "of dataset/model_ready (or clone the project repo to /content and "
            "set DATASET to its dataset/model_ready directory)."
        )
else:
    print("REVIEW MODE: completed retrospective min-val-loss experiment (epoch 37, test COCO mAP 0.7031666802); no Drive/GPU accessed.")


## 2 · Locked libraries and GPU

For selection/reproduction: pin `ultralytics==8.4.189` and `pycocotools==2.0.11`. Colab retains compatible torch/CUDA wheels and records installed package versions. A fresh GPU smoke test is still required.

In [ ]:
from importlib.metadata import version
import subprocess
import sys
import tempfile

# The project requirements.txt documents the Kaggle environment. Colab keeps
# its compatible torch/torchvision/numpy/Pillow wheels rather than forcing
# Kaggle's platform-specific binaries into the Colab runtime.
PINS = {"ultralytics": "8.4.189", "pycocotools": "2.0.11"}
if MODE in ("select", "reproduce"):
    subprocess.run([
        sys.executable, "-c",
        "import torch; assert torch.cuda.is_available(), 'Select GPU runtime in Colab'; "
        "print(torch.__version__, torch.cuda.get_device_name(0))"
    ], check=True)
    with tempfile.TemporaryDirectory() as setup_dir:
        constraints = Path(setup_dir) / "constraints.txt"
        keep = ("torch", "torchvision", "numpy", "Pillow")
        constraints.write_text("".join(f"{pkg}=={version(pkg)}\n" for pkg in keep))
        result = subprocess.run([
            sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
            "-c", str(constraints), *[f"{pkg}=={pin}" for pkg, pin in PINS.items()]
        ], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(result.stdout)
    if result.returncode:
        raise RuntimeError("Dependency installation failed; inspect the pip output above")
    subprocess.run([
        sys.executable, "-c",
        "import torch, torchvision, cv2; from ultralytics import YOLO; "
        "from pycocotools.cocoeval import COCOeval; assert torch.cuda.is_available()"
    ], check=True)
else:
    print("REVIEW MODE: dependency installation skipped.")


## 3 · Dataset validation and faithful 640×640 RGB bicubic geometry

Hash all inputs and labels; category IDs must be 1–52. Scene-level leakage is not resolved by exact SHA-256 checks.

In [ ]:
"""Read-only COCO dataset validation and temporary square YOLO inputs."""
from collections import defaultdict
from contextlib import contextmanager
import hashlib
from io import BytesIO
import json
import math
from pathlib import Path
from tempfile import TemporaryDirectory
import unicodedata

from PIL import Image

SIZE = 640
SPLITS = ("train", "validation", "test")
CLASS_IDS = list(range(1, 53))
DASHES = str.maketrans("–—−‑‐", "-----")
# The current repository and audited Drive split have these same image bytes.
HISTORICAL_SPLIT = (
    "3a7658a6224eb41b3be7c949ddc910295a56c7a93e09df6d0a94b65bce5129a8",
    "45ab837b8583eb70c3dc031a623a138a7c686ad8f8245d7ca36b0e80bf3ba09e",
    "ef238cafa4e72fc3e521eaeb419b8fdb2f2266152599a341c4686572c3dd09db",
)


def canonical_name(name):
    return " ".join(unicodedata.normalize("NFKC", name).translate(DASHES).split()).casefold()


def sha256(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8-sig"))


def fingerprint(data):
    encoded = json.dumps(data, sort_keys=True, separators=(",", ":")).encode()
    return hashlib.sha256(encoded).hexdigest()


class Dataset:
    def __init__(self, root, groups_path=None):
        self.root = root.resolve()
        self.images, self.boxes, self.annotations, self.digests = {}, {}, {}, {}
        manifest, owners, overlaps = {}, {}, set()
        groups = read_json(groups_path) if groups_path else None
        if groups is not None and (not isinstance(groups, dict) or set(groups) != set(SPLITS)):
            raise ValueError("Group manifest must map train, validation and test to file/group maps")
        self.names = None
        group_owners = {}
        for split in SPLITS:
            self.annotations[split] = self.root / "annotations" / f"instances_{split}.json"
            annotation_bytes = self.annotations[split].read_bytes()
            doc = json.loads(annotation_bytes.decode("utf-8-sig"))
            categories = sorted(doc["categories"], key=lambda row: row["id"])
            if [row["id"] for row in categories] != CLASS_IDS:
                raise ValueError(f"{split}: expected COCO category IDs 1 through 52")
            names = [row["name"] for row in categories]
            if self.names is None:
                self.names = names
            if list(map(canonical_name, names)) != list(map(canonical_name, self.names)):
                raise ValueError(f"{split}: inconsistent category names")

            images = doc["images"]
            ids = {image["id"]: image for image in images}
            filenames = {image["file_name"] for image in images}
            if not images or len(ids) != len(images) or len(filenames) != len(images):
                raise ValueError(f"{split}: empty split or duplicate image ID/file name")
            if groups is not None and (not isinstance(groups[split], dict)
                                       or set(groups[split]) != filenames):
                raise ValueError(f"{split}: group manifest coverage differs from COCO")

            hashes = {}
            for image in images:
                name = image["file_name"]
                if (Path(name).name != name or name in {".", ".."}
                        or type(image["id"]) is not int or min(image["width"], image["height"]) <= 0):
                    raise ValueError(f"{split}: invalid image record {image}")
                digest = sha256(self.path(split, image))
                if owners.setdefault(digest, split) != split:
                    overlaps.add(digest)
                hashes[name] = digest
                if groups is not None:
                    group = groups[split][name]
                    if not isinstance(group, str) or not group.strip():
                        raise ValueError(f"{split}/{name}: missing scene group")
                    if group_owners.setdefault(group, split) != split:
                        raise ValueError(f"Scene group crosses splits: {group}")
            boxes = defaultdict(list)
            for ann in doc["annotations"]:
                image = ids[ann["image_id"]]
                width, height = image["width"], image["height"]
                x, y, w, h = ann["bbox"]
                if (ann["category_id"] not in CLASS_IDS or ann.get("iscrowd", 0)
                        or not all(math.isfinite(v) for v in (x, y, w, h))
                        or min(x, y) < 0 or min(w, h) <= 0
                        or x + w > width + 1e-5 or y + h > height + 1e-5):
                    raise ValueError(f"{split}: unsupported COCO annotation {ann['id']}")
                boxes[ann["image_id"]].append(
                    (ann["category_id"] - 1, (x + w / 2) / width, (y + h / 2) / height,
                     w / width, h / height)
                )
            if not boxes or len({ann["id"] for ann in doc["annotations"]}) != len(doc["annotations"]):
                raise ValueError(f"{split}: missing annotations or repeated annotation IDs")
            manifest[split] = {
                "images": len(images), "annotations": len(doc["annotations"]),
                "annotation_sha256": hashlib.sha256(annotation_bytes).hexdigest(),
                "image_manifest_sha256": fingerprint(hashes),
            }
            self.images[split], self.boxes[split], self.digests[split] = images, boxes, hashes

        split_hashes = tuple(fingerprint(sorted(self.digests[split].values())) for split in SPLITS)
        if groups is not None and split_hashes == HISTORICAL_SPLIT:
            raise ValueError("The known scene-overlapping split is historical even with group IDs")
        if groups is not None and overlaps:
            raise ValueError(f"Found {len(overlaps)} exact image hashes across grouped splits")
        self.summary = {
            "track": "comparison" if groups is not None else "historical",
            "dataset_sha256": fingerprint({"splits": manifest, "groups": groups}),
            "splits": manifest, "group_count": len(group_owners),
            "exact_cross_split_hashes": len(overlaps),
        }

    def path(self, split, image):
        return self.root / "images" / split / image["file_name"]

    def square(self, split, image):
        path = self.path(split, image)
        data = path.read_bytes()
        if hashlib.sha256(data).hexdigest() != self.digests[split][image["file_name"]]:
            raise ValueError(f"Image changed since preflight: {path}")
        with Image.open(BytesIO(data)) as source:
            if source.size != (image["width"], image["height"]):
                raise ValueError(f"Image dimensions disagree with COCO: {self.path(split, image)}")
            return source.convert("RGB").resize((SIZE, SIZE), Image.Resampling.BICUBIC)

    @contextmanager
    def stage(self):
        with TemporaryDirectory(prefix="vgu-yolo-") as directory:
            stage = Path(directory)
            for split in ("train", "validation"):
                (stage / "images" / split).mkdir(parents=True)
                (stage / "labels" / split).mkdir(parents=True)
                for image in self.images[split]:
                    stem = str(image["id"])
                    self.square(split, image).save(
                        stage / "images" / split / f"{stem}.png", compress_level=1
                    )
                    lines = [
                        f"{category} {cx:.10f} {cy:.10f} {w:.10f} {h:.10f}"
                        for category, cx, cy, w, h in self.boxes[split][image["id"]]
                    ]
                    (stage / "labels" / split / f"{stem}.txt").write_text(
                        "\n".join(lines), encoding="utf-8"
                    )
            config = stage / "data.yaml"
            # JSON is valid YAML and preserves category order and the absolute stage path.
            config.write_text(json.dumps({
                "path": str(stage), "train": "images/train", "val": "images/validation",
                "nc": len(self.names), "names": self.names,
            }), encoding="utf-8")
            yield config

if MODE in ("select", "reproduce"):
    dataset = Dataset(DATASET)
    print(json.dumps(dataset.summary, indent=2))
    record = {
        "track": "historical", "purpose": "colab_pilot", "phase2a_complete": False,
        "dataset_sha256": dataset.summary["dataset_sha256"],
    }
else:
    print("REVIEW MODE: dataset integrity class loaded; no source images accessed.")


## 4 · Freeze the 50-epoch training run

`MODE="select"` only verifies the existing checkpoint manifest; **never calls `.train()`**. `MODE="reproduce"` can train/resume if missing. Do not delete or overwrite the original `selection/` and `test/` artifacts.

In [ ]:
from types import SimpleNamespace
import platform

EPOCHS = 50
SEED = 42
NMS_IOU = 0.7
TRAIN_ARGS = {
    "epochs": EPOCHS, "imgsz": SIZE, "seed": SEED, "deterministic": True,
    "optimizer": "AdamW", "lr0": 0.000179, "weight_decay": 0.0005, "nbs": 8,
    "amp": True, "rect": False, "cache": False, "patience": 0,
    "save_period": 1, "val": False, "plots": False, "resume": False,
    "mosaic": 0.0, "mixup": 0.0, "copy_paste": 0.0, "cutmix": 0.0,
    "degrees": 0.0, "translate": 0.0, "scale": 0.0, "shear": 0.0,
    "perspective": 0.0, "flipud": 0.0, "fliplr": 0.0,
    "hsv_h": 0.0, "hsv_s": 0.0, "hsv_v": 0.0, "augmentations": [],
}

def write_new(path, value):
    """Create-only JSON: never silently overwrite experiment provenance."""
    with path.open("x", encoding="utf-8") as stream:
        json.dump(value, stream, ensure_ascii=False, indent=2, allow_nan=False)
        stream.write("\n")


def atomic_json(path, value):
    """Crash-safe JSON replacement (used for resumable derived results only)."""
    import os
    from uuid import uuid4
    tmp = path.with_name(f".{path.name}.{uuid4().hex}.tmp")
    try:
        with tmp.open("x", encoding="utf-8") as stream:
            json.dump(value, stream, ensure_ascii=False, indent=2, allow_nan=False)
            stream.write("\n")
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(tmp, path)
    finally:
        tmp.unlink(missing_ok=True)


def runtime():
    import torch

    for name, expected in PINS.items():
        if version(name) != expected:
            raise RuntimeError(f"Expected {name}=={expected} from requirements.txt")
    return {
        "python": platform.python_version(), "platform": platform.platform(),
        "packages": {name: version(name) for name in (*PINS, "torch", "torchvision", "Pillow")},
        "cuda": torch.version.cuda,
        "visible_cuda_devices": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    }


def checked_run(dataset, args):
    run = args.run.resolve()
    if (run.is_relative_to(dataset.root) or dataset.root.is_relative_to(run)
            or {"yolo11s_v1", "rtdetr_r50vd_comparison_50ep",
                "faster_rcnn_comparison_50ep"}.intersection(run.parts)):
        raise ValueError("Use a dedicated YOLO11s run directory outside the dataset and legacy runs")
    return run


def checkpoint(run, epoch):
    # Ultralytics save_period uses zero-based epoch filenames.
    return run / "training" / "weights" / f"epoch{epoch - 1}.pt"


def train(dataset, args, record):
    run = checked_run(dataset, args)
    weights = args.weights.resolve(strict=True)
    if weights.name != "yolo11s.pt" or not weights.is_file():
        raise ValueError("Supply the official COCO-pretrained yolo11s.pt")
    environment = runtime()
    if int(args.device) >= len(environment["visible_cuda_devices"]):
        raise RuntimeError("Training needs the requested single CUDA device")

    from ultralytics import YOLO

    model = YOLO(str(weights), task="detect")
    if len(model.names) != 80:
        raise ValueError("Expected COCO-pretrained YOLO11s with 80 initializer classes")
    run.mkdir(parents=True, exist_ok=False)
    options = {**TRAIN_ARGS, "batch": args.batch, "device": args.device, "workers": args.workers}
    write_new(run / "run.json", {
        "dataset": dataset.summary, "protocol": record, "environment": environment,
        "initializer_sha256": sha256(weights), "train_options": options,
        "geometry": "640x640 RGB bicubic stretch", "checkpoint_rule": "min offline validation loss, earliest tie",
    })
    with dataset.stage() as config:
        model.train(data=str(config), project=str(run), name="training", exist_ok=False, **options)
    candidates = [checkpoint(run, epoch) for epoch in range(1, EPOCHS + 1)]
    if not all(path.is_file() for path in candidates):
        raise RuntimeError("The run did not produce the complete 50-epoch checkpoint series")
    write_new(run / "training_complete.json", {
        str(epoch): sha256(path) for epoch, path in enumerate(candidates, 1)
    })


def ensure_training(dataset, record):
    """Idempotent: detect completed, resume interrupted, or start a new run."""
    if (RUN / "training_complete.json").is_file():
        manifest = read_json(RUN / "training_complete.json")
        if set(manifest) != {str(k) for k in range(1, EPOCHS + 1)}:
            raise RuntimeError("Training manifest has an unexpected epoch set")
        if not all(checkpoint(RUN, e).is_file() for e in range(1, EPOCHS + 1)):
            raise RuntimeError("Training manifest exists but one or more checkpoints are missing")
        print(f"50-epoch training already completed: {RUN}")
        return

    if RUN.exists():
        from ultralytics import YOLO
        candidates = [checkpoint(RUN, e) for e in range(1, EPOCHS + 1)]
        missing = [p for p in candidates if not p.is_file()]
        if missing:
            last = RUN / "training" / "weights" / "last.pt"
            if not last.is_file():
                raise FileNotFoundError(f"Cannot resume; missing last.pt: {last}")
            runtime()
            with dataset.stage() as config:
                model = YOLO(str(last))
                saved_epoch = model.ckpt.get("epoch")
                optimizer = model.ckpt.get("optimizer")
                if not isinstance(saved_epoch, int) or optimizer is None:
                    raise RuntimeError("last.pt lacks the epoch/optimizer needed to resume")
                if not (0 <= saved_epoch < EPOCHS - 1):
                    raise RuntimeError("Checkpoint is already at/after final epoch")
                print(f"Resuming completed epoch {saved_epoch + 1} / {EPOCHS}")
                model.train(resume=True, data=str(config), device="0",
                            batch=BATCH, workers=2, save_period=1)
        if any(not p.is_file() for p in candidates):
            raise RuntimeError("Training is incomplete; inspect missing epoch checkpoints")
        # 50 checkpoints may exist even when training_complete.json was not
        # written (for example, Colab interrupted after epoch 50). Don't
        # resume from a finalized optimizer-stripped last.pt in that case.
        write_new(RUN / "training_complete.json", {
            str(e): sha256(p) for e, p in enumerate(candidates, 1)
        })
        print("Training verified: 50 epoch checkpoints saved.")
        return

    from ultralytics.utils.downloads import attempt_download_asset
    WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
    attempt_download_asset(str(WEIGHTS))
    train(dataset, SimpleNamespace(run=RUN, weights=WEIGHTS, batch=BATCH,
                                   device="0", workers=2), record)


if MODE == "reproduce":
    ensure_training(dataset, record)
elif MODE == "select":
    # Explicitly read-only with respect to the training state.
    trained_run_file = RUN / "run.json"
    completed = RUN / "training_complete.json"
    if not trained_run_file.is_file() or not completed.is_file():
        raise FileNotFoundError("Existing 50-epoch run is required in select mode")
    manifest = read_json(completed)
    if set(manifest) != {str(epoch) for epoch in range(1, EPOCHS + 1)}:
        raise RuntimeError("Expected 50 epoch hashes in existing training manifest")
    if not all(checkpoint(RUN, epoch).is_file() for epoch in range(1, EPOCHS + 1)):
        raise RuntimeError("One or more historical epoch checkpoints are missing")
    print("SELECT MODE: 50 saved checkpoints found. Training will NOT run.")
else:
    print("REVIEW MODE: no weights accessed; historical training finished 50 epochs.")


## 5 · OFFLINE VALIDATION LOSS — selection of all 50 frozen checkpoints

This section computes gain-weighted YOLO `box_loss`, `cls_loss`, `dfl_loss` for all 50 epochs on **validation images only**. The 255 resized 640×640 images are generated **once** into a temporary directory (not the 2,297 train images). Use an Ultralytics YOLO dataloader with `rect=False`, `augment=False` and FP16 autocast as recorded. Accumulate detached per-batch means and take their arithmetic mean (Ultralytics training-validation convention).

The history is persisted after each epoch in `selection_min_val_loss_v1/val_loss_history.json`, allowing resume on interruption. Ties choose the earlier epoch. Legacy `selection/selected.json` remains intact and is **not** read by this checkpoint selection.

**Requires GPU and all original checkpoints. Changing batch, precision, geometry, loss gains or dataloader changes the selection protocol and invalidates cached history.**

In [ ]:
import numpy as np
"""Common COCO bbox scoring for model prediction JSON in original image pixels."""
from collections import defaultdict
from contextlib import redirect_stdout
from io import StringIO
import json
import math
from pathlib import Path

SCORE_THRESHOLD = 0.001
IMAGE_BUDGET = 100
MAX_DETS = [1, 10, 100]
METRICS = (
    "map", "map_50", "map_75", "map_small", "map_medium", "map_large",
    "mar_1", "mar_10", "mar_100", "mar_small", "mar_medium", "mar_large",
)


class CocoScorer:
    def __init__(self, annotations: Path):
        from pycocotools.coco import COCO

        with redirect_stdout(StringIO()):
            self.truth = COCO(str(annotations))
        if not self.truth.imgs or not self.truth.cats or not self.truth.anns:
            raise ValueError("Scoring requires images, categories and annotated ground truth")

    def score(self, predictions):
        from pycocotools.coco import COCO
        from pycocotools.cocoeval import COCOeval

        if not isinstance(predictions, list):
            raise ValueError("Predictions must be a COCO JSON list")
        by_image = defaultdict(list)
        for row in predictions:
            image_id, category_id = row["image_id"], row["category_id"]
            score, bbox = row["score"], row["bbox"]
            if (type(image_id) is not int or image_id not in self.truth.imgs
                    or type(category_id) is not int or category_id not in self.truth.cats):
                raise ValueError(f"Prediction has an unknown image/category ID: {row}")
            if (not math.isfinite(score) or not 0 <= score <= 1 or len(bbox) != 4
                    or not all(math.isfinite(v) for v in bbox) or min(bbox[2:]) <= 0):
                raise ValueError(f"Prediction has invalid score or bbox: {row}")
            if score >= SCORE_THRESHOLD:
                by_image[image_id].append({
                    "image_id": image_id, "category_id": category_id,
                    "score": score, "bbox": list(bbox),
                })
        retained = [
            row for image_id in sorted(by_image)
            for row in sorted(by_image[image_id], key=lambda row: -row["score"])[:IMAGE_BUDGET]
        ]

        with redirect_stdout(StringIO()):
            if retained:
                # loadRes mutates its input entries by adding evaluation-only fields.
                detected = self.truth.loadRes([row.copy() for row in retained])
            else:
                detected = COCO()
                detected.dataset = {
                    "images": self.truth.dataset["images"],
                    "categories": self.truth.dataset["categories"],
                    "annotations": [],
                }
                detected.createIndex()
            evaluator = COCOeval(self.truth, detected, "bbox")
            evaluator.params.maxDets = MAX_DETS.copy()
            evaluator.evaluate()
            evaluator.accumulate()
            evaluator.summarize()

        precision = evaluator.eval["precision"]
        per_class = {}
        for i, category_id in enumerate(evaluator.params.catIds):
            values = precision[:, :, i, 0, -1]
            valid = values[values >= 0]
            per_class[str(int(category_id))] = {
                "name": self.truth.cats[int(category_id)]["name"],
                "map": float(valid.mean()) if valid.size else None,
            }
        return retained, {
            **dict(zip(METRICS, map(float, evaluator.stats))),
            "evaluated_images": len(self.truth.imgs), "predictions": len(retained),
            "score_threshold": SCORE_THRESHOLD, "max_dets": MAX_DETS,
            "global_detections_per_image": IMAGE_BUDGET, "per_class": per_class,
        }



def to_coco(image, coords, score, label):
    x1, y1, x2, y2 = coords
    sx, sy = image["width"] / SIZE, image["height"] / SIZE
    if x2 <= x1 or y2 <= y1:
        return None
    return {
        "image_id": image["id"], "category_id": int(label) + 1,
        "bbox": [x1 * sx, y1 * sy, (x2 - x1) * sx, (y2 - y1) * sy],
        "score": score,
    }


def bgr(dataset, split, image):
    return np.ascontiguousarray(np.asarray(dataset.square(split, image))[:, :, ::-1])


@contextmanager
def cached_validation(dataset):
    # Read Drive only once across 50 candidate evaluations. Cache is local and disk-backed.
    with TemporaryDirectory(prefix="vgu-yolo-val-") as directory:
        with (Path(directory) / "pixels.bgr").open("w+b") as stream:
            for image in dataset.images["validation"]:
                stream.write(bgr(dataset, "validation", image).tobytes())
        pixels = np.memmap(
            Path(directory) / "pixels.bgr", dtype=np.uint8, mode="r",
            shape=(len(dataset.images["validation"]), SIZE, SIZE, 3),
        )
        try:
            yield pixels
        finally:
            del pixels


def predict(dataset, split, weights, args, cache=None):
    from ultralytics import YOLO

    model = YOLO(str(weights), task="detect")
    if [canonical_name(model.names[i]) for i in sorted(model.names)] != [
        canonical_name(name) for name in dataset.names
    ]:
        raise ValueError("Checkpoint class order does not match the prepared COCO categories")
    images = dataset.images[split]
    predictions = []
    for start in range(0, len(images), args.eval_batch):
        batch = images[start:start + args.eval_batch]
        pixels = ([bgr(dataset, split, image) for image in batch] if cache is None
                  else list(cache[start:start + len(batch)]))
        results = model.predict(
            source=pixels, imgsz=SIZE, device=args.device, rect=False,
            conf=float(np.nextafter(np.float32(SCORE_THRESHOLD), np.float32(0))),
            iou=NMS_IOU, max_det=IMAGE_BUDGET, agnostic_nms=False,
            quantize=32, verbose=False, save=False,
        )
        if len(results) != len(batch):
            raise RuntimeError("YOLO did not return one result per image")
        for image, result in zip(batch, results):
            if result.orig_shape != (SIZE, SIZE):
                raise RuntimeError(f"Unexpected YOLO coordinate frame: {result.orig_shape}")
            for x1, y1, x2, y2, score, label in result.boxes.data.cpu().tolist():
                row = to_coco(image, (x1, y1, x2, y2), score, label)
                if row is not None:
                    predictions.append(row)
    return predictions


def trained_run(dataset, args, record):
    run = checked_run(dataset, args)
    metadata = read_json(run / "run.json")
    if metadata["dataset"] != dataset.summary or metadata["protocol"] != record:
        raise ValueError("Run dataset or protocol differs from the current inputs")
    return run, read_json(run / "training_complete.json")


def scorer_for(dataset, split):
    annotation = dataset.annotations[split]
    if sha256(annotation) != dataset.summary["splits"][split]["annotation_sha256"]:
        raise ValueError(f"{split} annotations changed since preflight")
    return CocoScorer(annotation)




# Offline checkpoint selection: minimize YOLO gain-weighted validation loss.
# This is a newly declared analysis applied retrospectively to the archived 50 epochs.
LOSS_DIR_NAME = "selection_min_val_loss_v1"
TEST_DIR_NAME = "test_min_val_loss_v1"
LOSS_BATCH = 8
LOSS_WORKERS = 2
LOSS_AMP = True  # match the historical AMP=True run; model weights remain fp32
LOSS_KEYS = ("box_loss", "cls_loss", "dfl_loss")
LOSS_PROTOCOL_VERSION = "vgu-yolo11s-min-val-loss-1"


def choose_min_loss(history):
    """Select minimum weighted validation loss; earlier epoch breaks exact ties."""
    if not history:
        raise ValueError("Validation loss history is empty")
    if any(not math.isfinite(float(row["total_val_loss"])) for row in history):
        raise ValueError("Validation losses must be finite")
    return min(history, key=lambda row: (float(row["total_val_loss"]), int(row["epoch"])))


@contextmanager
def stage_loss_validation(dataset):
    """Create ONLY the 255 validation PNGs + their YOLO labels in temporary disk.

    Avoid preparing all 2297 training images for an evaluation-only correction.
    Square images are already bicubic stretched; there must be NO added padding.
    """
    with TemporaryDirectory(prefix="vgu-yolo-minloss-") as directory:
        root = Path(directory)
        images = root / "images" / "validation"
        labels = root / "labels" / "validation"
        images.mkdir(parents=True)
        labels.mkdir(parents=True)
        for image in dataset.images["validation"]:
            stem = str(image["id"])
            dataset.square("validation", image).save(images / f"{stem}.png", compress_level=1)
            lines = [
                f"{class_id} {cx:.10f} {cy:.10f} {w:.10f} {h:.10f}"
                for class_id, cx, cy, w, h in dataset.boxes["validation"][image["id"]]
            ]
            (labels / f"{stem}.txt").write_text("\n".join(lines), encoding="utf-8")
        if len(list(images.glob("*.png"))) != len(dataset.images["validation"]):
            raise RuntimeError("Validation staging did not prepare every image")
        yield images


def make_loss_loader(dataset, images_path):
    """Ultralytics 8.4.189 detection validation dataloader, fixed-square/no-pad."""
    import torch
    from ultralytics.cfg import DEFAULT_CFG, get_cfg
    from ultralytics.data.build import build_yolo_dataset, build_dataloader

    cfg = get_cfg(DEFAULT_CFG, overrides={
        "task": "detect", "imgsz": SIZE, "batch": LOSS_BATCH,
        "rect": False, "cache": False, "split": "val", "fraction": 1.0,
        "classes": None, "workers": LOSS_WORKERS,
    })
    names = {i: name for i, name in enumerate(dataset.names)}
    ds = build_yolo_dataset(
        cfg, str(images_path), batch=LOSS_BATCH,
        data={"nc": len(names), "names": names, "channels": 3},
        mode="val", rect=False, stride=32, fraction=1.0,
    )
    if ds.rect:
        raise RuntimeError("Unexpected rectangular mode: 640x640 loss must have no added padding")
    if len(ds) != len(dataset.images["validation"]):
        raise RuntimeError(f"Validation loader has {len(ds)} images, expected {len(dataset.images['validation'])}")
    loader = build_dataloader(
        ds, batch=LOSS_BATCH, workers=LOSS_WORKERS, shuffle=False,
        rank=-1, drop_last=False, pin_memory=True, device=torch.device("cuda:0"),
    )
    return loader


def loss_on_checkpoint(weights, loader, dataset):
    """Compute exactly three YOLO11s gain-weighted loss components, eval-mode.

    Follows Ultralytics validator-style aggregation: sum detached component
    means over dataloader batches, then divide by number of batches. This is
    *not* COCO AP and does not use any test images or labels.
    """
    import torch
    from ultralytics import YOLO

    yolo = YOLO(str(weights), task="detect")
    if [canonical_name(yolo.names[i]) for i in sorted(yolo.names)] != [
        canonical_name(n) for n in dataset.names
    ]:
        raise ValueError("Checkpoint class order does not match COCO validation classes")
    from types import SimpleNamespace
    from ultralytics.cfg import DEFAULT_CFG_DICT

    # IMPORTANT: YOLO(weights) leaves .model.args as a dict of inference-safe
    # overrides. DetectionModel.loss expects attribute-style training hyperparams;
    # reconstruct them from the SAVED training checkpoint, not new defaults.
    saved_args = yolo.ckpt.get("train_args") if yolo.ckpt is not None else None
    if not isinstance(saved_args, dict):
        raise RuntimeError("Checkpoint does not contain train_args for loss gains")
    full_args = {**DEFAULT_CFG_DICT, **saved_args}
    model = yolo.model.to("cuda:0").float().eval()
    model.args = SimpleNamespace(**full_args)
    # An old criterion may carry CPU tensors; rebuild on current GPU.
    model.criterion = None
    if not all(hasattr(model.args, key) for key in ("box", "cls", "dfl")):
        raise RuntimeError("Checkpoint does not expose YOLO loss gains")

    # Critical: model.eval() is required for reproducible BatchNorm behavior;
    # loss.parse_output handles the (inference, raw-features) tuple in eval mode.
    component_sums = {k: 0.0 for k in LOSS_KEYS}
    num_batches, num_images = 0, 0
    with torch.no_grad():
        for batch in loader:
            batch = {
                key: (value.to("cuda:0", non_blocking=True) if isinstance(value, torch.Tensor) else value)
                for key, value in batch.items()
            }
            if tuple(batch["img"].shape[-2:]) != (SIZE, SIZE):
                raise RuntimeError(f"Unexpected loss input image geometry: {batch['img'].shape}")
            num_images += int(batch["img"].shape[0])
            batch["img"] = batch["img"].float().div_(255.0)
            with torch.autocast(device_type="cuda", enabled=LOSS_AMP):
                _total, items = model.loss(batch)
            if set(items) != set(LOSS_KEYS):
                raise RuntimeError(f"Unexpected checkpoint loss components: {list(items)}")
            for key in LOSS_KEYS:
                value = float(items[key].detach().float().item())
                if not math.isfinite(value) or value < 0:
                    raise ValueError(f"Invalid {key}: {value}")
                component_sums[key] += value
            num_batches += 1
    if not num_batches or num_images != len(dataset.images["validation"]):
        raise RuntimeError(f"Incorrect loss loader coverage: {num_images} images")
    means = {key: component_sums[key] / num_batches for key in LOSS_KEYS}
    total = sum(means.values())
    del yolo, model
    torch.cuda.empty_cache()
    return {**means, "total_val_loss": total,
            "validation_images": num_images, "batches": num_batches}


def min_loss_protocol(dataset, completed):
    return {
        "version": LOSS_PROTOCOL_VERSION,
        "selection": "argmin(box_loss + cls_loss + dfl_loss), earliest epoch tie",
        "criterion": "Ultralytics 8.4.189 YOLO11s DetectionModel.loss",
        "aggregation": "mean of per-batch loss components (Ultralytics trainer validator convention)",
        "labels": "COCO train=0-based YOLO labels for 52 classes; validation only",
        "geometry": "RGB bicubic stretch to 640x640; rect=False; no additional padding",
        "eval_mode": True, "amp": LOSS_AMP, "batch": LOSS_BATCH,
        "workers": LOSS_WORKERS, "device": "cuda:0", "loss_keys": list(LOSS_KEYS),
        "dataset_sha256": dataset.summary["dataset_sha256"],
        "training_manifest_fingerprint": fingerprint(completed),
        "test_used_for_selection": False,
        "historical_test_previously_viewed": True,
        "packages": {name: version(name) for name in ("ultralytics", "torch", "torchvision", "numpy", "Pillow")},
        "cuda": torch.version.cuda,
    }


def verify_minloss_selection(dataset, completed, output):
    history = read_json(output / "val_loss_history.json")
    if len(history) != EPOCHS or [int(r["epoch"]) for r in history] != list(range(1, EPOCHS+1)):
        raise RuntimeError("Loss history must contain all 50 epochs in order")
    best = choose_min_loss(history)
    epoch = int(best["epoch"])
    selected = read_json(output / "selected_min_val_loss.json")
    if (selected.get("checkpoint_sha256") != completed[str(epoch)]
            or int(selected.get("epoch", -1)) != epoch
            or abs(float(selected.get("total_val_loss", float("nan"))) - float(best["total_val_loss"])) > 1e-10):
        raise RuntimeError("Selected min-loss checkpoint does not match manifest/history")
    file = checkpoint(RUN, epoch)
    if not file.is_file() or sha256(file) != completed[str(epoch)]:
        raise RuntimeError("Selected checkpoint hash failed")
    print(f"MIN-LOSS CHECKPOINT VERIFIED: epoch {epoch}/{EPOCHS}; total_val_loss={best['total_val_loss']:.8f}")
    return selected


def ensure_minloss_selection(dataset, record):
    if not torch.cuda.is_available():
        raise RuntimeError("Minimum-validation-loss selection requires a CUDA runtime")
    runtime()  # check Ultralytics+COCOeval pins after reconnect
    run, completed = trained_run(dataset, SimpleNamespace(run=RUN), record)
    output = run / LOSS_DIR_NAME
    output.mkdir(exist_ok=True)
    protocol = min_loss_protocol(dataset, completed)
    pfile = output / "protocol.json"
    if pfile.is_file():
        if read_json(pfile) != protocol:
            raise RuntimeError("Existing loss selection uses different settings: create a NEW output folder")
    else:
        write_new(pfile, protocol)
    if (output / "selected_min_val_loss.json").is_file():
        return verify_minloss_selection(dataset, completed, output)

    history_path = output / "val_loss_history.json"
    history = read_json(history_path) if history_path.is_file() else []
    if len(history) > EPOCHS or [int(x["epoch"]) for x in history] != list(range(1,len(history)+1)):
        raise RuntimeError("Partial validation loss history is corrupt/non-contiguous")
    for row in history:
        if row.get("checkpoint_sha256") != completed[str(row["epoch"])]:
            raise RuntimeError("Existing history fingerprint does not match original checkpoints")
        if abs(sum(float(row[k]) for k in LOSS_KEYS) - float(row["total_val_loss"])) > 1e-8:
            raise RuntimeError("Existing history component sums mismatch")

    # The validation image stage and dataloader are created once, reused for 50 checkpoints.
    if len(history) < EPOCHS:
        with stage_loss_validation(dataset) as val_images:
            loader = make_loss_loader(dataset, val_images)
            for epoch in range(len(history)+1, EPOCHS+1):
                weight = checkpoint(run, epoch)
                if sha256(weight) != completed[str(epoch)]:
                    raise RuntimeError(f"Bad checkpoint hash epoch {epoch}")
                values = loss_on_checkpoint(weight, loader, dataset)
                row = {"epoch": epoch, "checkpoint_sha256": completed[str(epoch)], **values}
                history.append(row)
                atomic_json(history_path, history)
                print(f"Validation loss epoch {epoch:02}/{EPOCHS}: "
                      f"box={values['box_loss']:.6f} cls={values['cls_loss']:.6f} "
                      f"dfl={values['dfl_loss']:.6f} TOTAL={values['total_val_loss']:.6f}", flush=True)
            del loader

    best = choose_min_loss(history)
    epoch = int(best["epoch"])
    selection = {
        "epoch": epoch,
        "checkpoint_sha256": completed[str(epoch)],
        "total_val_loss": float(best["total_val_loss"]),
        "box_loss": float(best["box_loss"]),
        "cls_loss": float(best["cls_loss"]),
        "dfl_loss": float(best["dfl_loss"]),
        "selection_rule": protocol["selection"],
        "dataset_sha256": protocol["dataset_sha256"],
        "protocol_fingerprint": fingerprint(protocol),
        "selection_origin": "retrospective evaluation of original 50 epoch checkpoints",
    }
    write_new(output / "selected_min_val_loss.json", selection)
    return verify_minloss_selection(dataset, completed, output)


import torch
if MODE in ("select", "reproduce"):
    selected_min_loss = ensure_minloss_selection(dataset, record)
    print("Min-loss selected checkpoint:", checkpoint(RUN, int(selected_min_loss["epoch"])))
else:
    print("REVIEW MODE: archived min-loss selection: epoch 37/50, loss=2.315651791. Recorded in results/YOLO11s/min_val_loss/.")


## 6 · Retrospective COCO test with the min-val-loss-selected checkpoint

**Only after `selection_min_val_loss_v1/selected_min_val_loss.json` exists.** Output goes to `test_min_val_loss_v1/`; the original max-AP pilot `test/` files remain untouched. The new test evaluation is retrospective: the historical test split had already been inspected in the earlier pilot.


In [ ]:

from types import SimpleNamespace


def minloss_test_protocol(selected, dataset):
    return {
        "selection_rule": "minimum YOLO validation loss (box+cls+dfl), earliest tie",
        "selection_epoch": int(selected["epoch"]),
        "checkpoint_sha256": selected["checkpoint_sha256"],
        "dataset_sha256": dataset.summary["dataset_sha256"],
        "score_floor": SCORE_THRESHOLD, "nms_iou": NMS_IOU,
        "max_detections_per_image": IMAGE_BUDGET,
        "images": "test", "evaluation": "COCOeval bbox",
        "test_status": "RETROSPECTIVE -- historical test already inspected under prior checkpoint rule",
    }


def ensure_minloss_test(dataset, record):
    run, completed = trained_run(dataset, SimpleNamespace(run=RUN), record)
    selection = verify_minloss_selection(dataset, completed, run / LOSS_DIR_NAME)
    epoch = int(selection["epoch"])
    weight = checkpoint(run, epoch)
    protocol = minloss_test_protocol(selection, dataset)
    test_output = run / TEST_DIR_NAME
    test_output.mkdir(exist_ok=True)
    pfile = test_output / "protocol.json"
    if pfile.is_file():
        if read_json(pfile) != protocol:
            raise RuntimeError("Existing min-loss test artifacts use different checkpoint/config")
    else:
        write_new(pfile, protocol)

    pred_file = test_output / "test_predictions.json"
    metrics_file = test_output / "test_metrics.json"
    if metrics_file.is_file():
        results = read_json(metrics_file)
        if (results.get("checkpoint_sha256") != selection["checkpoint_sha256"]
                or results.get("dataset_sha256") != dataset.summary["dataset_sha256"]
                or int(results.get("selected_epoch", -1)) != epoch
                or results.get("selection_rule") != protocol["selection_rule"]):
            raise RuntimeError("Existing test metrics from another protocol/checkpoint")
        if not pred_file.is_file() or len(read_json(pred_file)) != results["predictions"]:
            raise RuntimeError("Existing test predictions are missing or inconsistent")
        print("Retrospective min-loss test metrics already complete; inference skipped")
        return results

    if pred_file.is_file():
        predictions = read_json(pred_file)
    else:
        if sha256(weight) != selection["checkpoint_sha256"]:
            raise RuntimeError("Selected model SHA-256 has changed")
        predictions = predict(dataset, "test", weight, SimpleNamespace(device="0", eval_batch=2))
        atomic_json(pred_file, predictions)

    _, metrics = scorer_for(dataset, "test").score(predictions)
    metrics.update({
        "split": "test", "selected_epoch": epoch,
        "checkpoint_sha256": selection["checkpoint_sha256"],
        "dataset_sha256": dataset.summary["dataset_sha256"],
        "selection_rule": protocol["selection_rule"],
        "test_status": protocol["test_status"],
        "track": dataset.summary["track"],
    })
    atomic_json(metrics_file, metrics)
    return metrics


if MODE in ("select", "reproduce"):
    matched_test_metrics = ensure_minloss_test(dataset, record)
    print("\nMIN-LOSS-SELECTED YOLO11s: RETROSPECTIVE TEST RESULTS")
    for key in ("map", "map_50", "map_75", "map_small", "map_medium", "map_large", "mar_100"):
        print(f"{key:15s}: {matched_test_metrics[key]:.6f}")
    print("Selected epoch:", matched_test_metrics["selected_epoch"])
    print("Saved under:", RUN / TEST_DIR_NAME)
else:
    matched_test_metrics = {
    "map": 0.7031666801599523,
    "map_50": 0.9240568340395987,
    "map_75": 0.8399927544914599,
    "map_small": 0.6871557962790006,
    "map_medium": 0.7512722453454844,
    "map_large": 0.8425181936300195,
    "mar_100": 0.7490599566034645,
    "evaluated_images": 639,
    "predictions": 2461
}
    print("REVIEW MODE: archived retrospective min-validation-loss test results (not recomputed)")
    for key in ("map", "map_50", "map_75", "map_small", "map_medium", "map_large", "mar_100"):
        print(f"{key:15s}: {matched_test_metrics[key]:.6f}")
    print("Selected epoch: 37 (epoch36.pt); selected by min validation loss")
    print("Predictions:", matched_test_metrics['predictions'])
    print("Independent COCOeval replay is available in scripts/YOLO11s/evaluate/replay_coco.py")


## 7 · Interpretation and reporting caveats

- The historical max-validation-AP pilot selected epoch 36/50 and obtained test COCO mAP@[.50:.95] = **0.7033492209**. This result is **preserved**, not re-labelled as min-loss.
- Minimum validation-loss selection **completed** in Colab on 11 Nov 2026: selected epoch **37/50** (`epoch36.pt`), validation loss **2.315651791**; retrospective test COCO mAP@[.50:.95] **0.7031666802** on **639 test images** with **2,461 predictions**. Review mode displays the archived outputs rather than recomputing them.
- RT-DETR R50vd and Faster R-CNN were reported as choosing minimum validation loss, but their losses use **architecture-specific objectives** and are not numerically comparable to YOLO's three gain-weighted components.
- YOLO `val=False` during the historical training, so `results.csv` contains no recorded validation loss. This new calculation is performed **post hoc** from the 50 saved model checkpoints.
- This revised selection protocol was applied **after the historical test was already viewed**, so its resulting test score is a **retrospective comparison**, not an untouched prospective holdout. A new independent test is required for a stronger confirmatory claim.
- Phase 2A scene-level leakage audit remains incomplete; at least 55 of 639 historical test images are linked to train scenes in reviewed evidence. The split is not leakage-free.
- Preserve the original `run.json`, `selection/` and `test/` artifacts; the revised selection is stored under `selection_min_val_loss_v1/` and `test_min_val_loss_v1/`. Do not mix their checkpoints/metrics in a single row.
- The revised loss-scoring execution path has been statically checked, **not run on actual Ultralytics checkpoints in this sandbox**. A one-checkpoint smoke test in Colab is required before committing official results.

- GitHub review mode displays archived results; `MODE="select"` uses the existing Drive checkpoints and will verify/resume without training. This user-supplied Colab run is a **retrospective** analysis.
